# 01 · Behaviour

Recognition performance in the continuous recognition task and its relation to image memorability.

* **Model** (paper: Results, *Behavioural results*; Methods, *Behavioural results*): binomial GLM (logit link)
  predicting a correct response from the interaction between ViT memorability and presentation number (1–3),
  with heteroscedasticity-consistent (HC0) standard errors. Trials without a response (omissions) are excluded.
* **Table 1**: correct / answered trials per participant and presentation.
* **Figure 1b**: distribution of memorability scores across images.

Inputs: `trials.pkl` per participant and the memorability scores (see `config.py`).
Outputs: `results/behaviour_glm.txt`, `results/table1_behaviour.csv`, `figures/fig1b_memorability_distribution.svg`.

With `USE_SHARED_DATA=1`, the trials are read from `behaviour.csv` in the shared dataset instead
(participants labelled sub-01 … sub-15), and outputs go to `results/from_shared_data/` and `figures/from_shared_data/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

import config
import preprocessing  # noqa: F401  (makes NumPy-2 pickles readable under NumPy 1.x)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load trials of all participants

In [ ]:
memorability_scores = pd.read_csv(config.MEMORABILITY_FILE).rename(columns={"image": "image_ID"})

if config.USE_SHARED_DATA:
    # One row per presentation, already restricted to image trials and numbered (see 05_export_shared_data)
    behaviour = pd.read_csv(config.SHARED_DATA_DIR / "behaviour.csv").rename(
        columns={"subject": "patient", "image_id": "image_ID", "presentation": "occurence",
                 "response": "last_response"})
    behaviour = behaviour.merge(memorability_scores[["image_ID", "vit_memorability"]], on="image_ID")
else:
    frames = []
    for patient_id in config.PATIENTS:
        trials = pd.read_pickle(config.SEEG_DATA_ROOT / patient_id / f"analysis_{config.TASK}" / "trials.pkl")
        df = trials.drop(columns=["pre_trial_ephys", "ephys", "channels"])
        df["patient"] = patient_id
        # Image trials only (inner merge on the image ID)
        df = df.merge(memorability_scores[["image_ID", "vit_memorability"]], on="image_ID")
        # Presentation number of each image: 1 = first, 2 = second, ...
        df["occurence"] = df.groupby("image_ID").cumcount() + 1
        frames.append(df)
    behaviour = pd.concat(frames, ignore_index=True)
behaviour["old"] = behaviour["last_response"].map({"new": 0, "old": 1})
print(behaviour.groupby("patient").size())

## Recognition accuracy as a function of memorability and presentation number

In [ ]:
df_sel = behaviour.loc[behaviour["occurence"] <= 3,
                       ["patient", "last_response", "vit_memorability", "old", "occurence"]].copy()
df_sel["occurence"] = df_sel["occurence"].astype("category")

n_presentations = len(df_sel)
df_sel = df_sel.dropna()  # removes omissions (no 'new'/'old' response)
print(f"Presentations: {n_presentations}, answered: {len(df_sel)}, omissions: {n_presentations - len(df_sel)}")

# Correct = 'new' on the first presentation, 'old' on later presentations
df_sel["correct_resp"] = (((df_sel["occurence"] == 1) & (df_sel["last_response"] == "new"))
                          | ((df_sel["occurence"] != 1) & (df_sel["last_response"] == "old"))).astype(int)

model = smf.glm("correct_resp ~ occurence:vit_memorability", data=df_sel, family=sm.families.Binomial())
result = model.fit(cov_type="HC0")
print(result.summary())
print(f"AIC: {result.aic}")

with open(config.RESULTS_DIR / "behaviour_glm.txt", "w") as f:
    f.write(f"Presentations: {n_presentations}, answered: {len(df_sel)}, "
            f"omissions: {n_presentations - len(df_sel)}\n\n")
    f.write(str(result.summary()))

## Table 1: performance per participant and presentation

In [ ]:
def fmt(correct, total):
    return f"{int(correct)}/{int(total)} ({correct / total:.2%})"


counts = df_sel.groupby(["patient", "occurence"], observed=True)["correct_resp"].agg(["sum", "size"])
table1 = counts.apply(lambda r: fmt(r["sum"], r["size"]), axis=1).unstack("occurence")
table1.columns = [f"Occurrence {c}" for c in table1.columns]

per_patient = df_sel.groupby("patient")["correct_resp"].agg(["sum", "size"])
table1["Overall"] = per_patient.apply(lambda r: fmt(r["sum"], r["size"]), axis=1)

per_occurrence = df_sel.groupby("occurence", observed=True)["correct_resp"].agg(["sum", "size"])
sum_row = {f"Occurrence {occ}": fmt(r["sum"], r["size"]) for occ, r in per_occurrence.iterrows()}
sum_row["Overall"] = fmt(df_sel["correct_resp"].sum(), len(df_sel))
table1.loc["Sum"] = pd.Series(sum_row)

table1.to_csv(config.RESULTS_DIR / "table1_behaviour.csv")
table1

## Figure 1b: distribution of memorability scores across images

In [ ]:
image_scores = behaviour.groupby("image_ID")["vit_memorability"].mean().reset_index()

sns.set_theme(style="darkgrid")
plt.figure(figsize=(5, 5), dpi=500)
sns.histplot(image_scores["vit_memorability"], kde=True, color=sns.dark_palette("#69d", reverse=True, n_colors=4)[1], bins=20)
plt.xlabel("ViT Memorability")
plt.ylabel("Frequency")
plt.xlim(0.5, 0.99)
plt.yticks(np.arange(0, 126, 25))
plt.ylim(0, 120)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "fig1b_memorability_distribution.svg", format="svg", bbox_inches="tight")
plt.show()